# ⏱️ 08 - Timing and Performance

In data science, performance matters. Code that's instant on 100 rows can take minutes on 10 million. Jupyter/IPython give you tools to **measure** how long code takes, find **where** the time goes, and check how much **memory** your data uses.

## ✅ Learning Goals
- Use `%time` and `%%time` for a quick, one-time measurement
- Use `%timeit` and `%%timeit` for reliable comparisons, and read their output
- Compare two approaches **fairly**: same input, same result, setup excluded
- Compare Python loops with NumPy **vectorized** operations
- Spot repeated work, and see how runtime grows with data size
- Find the slow part of a function with `%prun` (and `%lprun`)
- Measure memory with `nbytes` and `memory_usage`, and load data efficiently with Pandas

> **The core habit:** measure first, improve the bottleneck, then measure again.

## Setup
All imports go in the first cell (notebook 02). This notebook creates one data file, in `08-OutputFiles`.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('08-OutputFiles')
OUT.mkdir(exist_ok=True)

## 1. Why Measure?
Performance work should start with **evidence**, not a hunch:

```text
Feels slow → measure → find the bottleneck → improve it → measure again
```

Don't optimize code just because it **looks** slow. A loop may be fine for 100 values and painful for 10 million, and a clever-looking rewrite may not help the real workload at all.

Useful questions:
- Which cell takes the longest? Which function inside it?
- Does the runtime grow too quickly as the data grows?
- Is the code doing the **same work** over and over?
- Could NumPy or Pandas do the operation **in bulk** instead of one item at a time?

## 2. `%time`: One Quick Measurement
`%time` runs a statement **once** and reports how long it took. It works on a line with an assignment, too.

In [2]:
numbers = list(range(1_000_000))

%time total = sum(numbers)

CPU times: total: 15.6 ms
Wall time: 6.79 ms


The output shows two kinds of time:
- **Wall time** is real "clock on the wall" time, how long **you** waited. This is usually the number you care about.
- **CPU times** are how long the processor was actually busy. Wall time is much bigger than CPU time when the code is **waiting**, for example on a file, a website, or a slow disk.

✅ **Your Turn**: Use `%time` to measure summing one million integers with `np.sum(np.arange(1_000_000))`. Run the cell three times. Do you get the exact same time each run?

In [32]:
%time total_np = np.sum(np.arange(1_000_000))


CPU times: total: 0 ns
Wall time: 1.79 ms


No I do not get the same times each run

## 3. `%timeit`: Reliable Measurements
A single timing can be thrown off by other programs, background work, or one-time setup costs. `%timeit` runs an expression **many times** and reports the average. That's much more reliable for comparing two approaches.

In [33]:
numbers = list(range(1_000))
%timeit [x**2 for x in numbers]

32.3 μs ± 692 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Reading the Output
```text
41.2 μs ± 1.05 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
```
- **10,000 loops each**: in each run, the code was repeated 10,000 times. `%timeit` picks this count automatically, so fast code gets more loops.
- **7 runs**: that was done 7 separate times.
- **41.2 μs per loop**: on average, **one** execution took 41.2 microseconds.
- **± 1.05 μs**: the standard deviation, which shows how much the runs varied. Small is good.

| Unit | Meaning |
|---|---|
| s | second |
| ms | millisecond = 1/1,000 s |
| μs (or µs) | microsecond = 1/1,000,000 s |
| ns | nanosecond = 1/1,000,000,000 s |

### Saving the Result with `-o`
`-o` returns the timing as an object, so you can do math with it. For example, you can work out **how many times faster** one approach is:

In [34]:
t_comp = %timeit -o [n * 2 for n in range(10_000)]
t_map = %timeit -o list(map(lambda n: n * 2, range(10_000)))

print(f'The comprehension is {t_map.average / t_comp.average:.2f}x as fast as map + lambda')

272 μs ± 8.67 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
568 μs ± 7.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
The comprehension is 2.09x as fast as map + lambda


💡 For slow code, control the repetitions yourself: `%timeit -n 10 -r 3 ...` means 10 loops per run, 3 runs.

✅ **Your Turn**: Use `%timeit -o` to compare `sorted(data)` with `data.sort()` on a fresh copy, using `data = list(range(100_000, 0, -1))`. *Hint:* `data.sort()` changes the list in place, so time `data.copy().sort()` and `sorted(data.copy())` to make the comparison fair.

In [35]:
data = list(range(100_000, 0, -1))

t_sorted = %timeit -o sorted(data.copy())
t_sort = %timeit -o data.copy().sort()

print(f'list.sort() is {t_sorted.average / t_sort.average:.2f}x as fast as sorted()')

583 μs ± 8.42 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
423 μs ± 15.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
list.sort() is 1.38x as fast as sorted()


## 4. `%%time` and `%%timeit` for a Whole Cell
Put `%%time` or `%%timeit` as the **first line** of a cell to measure everything in it, not just one expression.

| Magic | Times... | Runs it... |
|---|---|---|
| `%time` | One line | Once |
| `%%time` | The whole cell | Once |
| `%timeit` | One line | Many times |
| `%%timeit` | The whole cell | Many times |

In [36]:
%%timeit
total = 0
for i in range(1_000_000):
    total += i

27.5 ms ± 1.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


⚠️ **Gotcha:** variables created inside a `%%timeit` cell **don't exist afterwards**. The cell ran in a temporary space, many times over. `%%time` runs your code normally, so its variables stay:

In [37]:
%%time
squares = [value ** 2 for value in range(100_000)]

CPU times: total: 15.6 ms
Wall time: 5.59 ms


In [38]:
len(squares)   # works, because %%time keeps variables. After %%timeit this would be a NameError

100000

## 5. Comparing Fairly
A timing comparison is only meaningful when both versions:
- use the **same input** (and the same size),
- do the **same work**, with no extra setup hiding in one of them,
- produce the **same result** (check it!),
- and are measured over **multiple runs** (`%timeit`).

**Unfair:** suppose your data starts as a list. The first line below also **converts** the list to an array every time, and the second doesn't. The difference is huge, and it has nothing to do with squaring:

In [39]:
data_list = list(range(1_000_000))
data_array = np.array(data_list)

%timeit np.array(data_list) ** 2    # converts the list AND squares it
%timeit data_array ** 2             # only squares it

32.5 ms ± 1.41 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.59 ms ± 102 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**Fair:** create the input **once**, outside the timing, then time only the operation you care about. And before comparing speed, confirm that both versions give the same answer:

In [40]:
values_list = list(range(1_000_000))
values_array = np.arange(1_000_000)

loop_result = [v ** 2 for v in values_list]
numpy_result = values_array ** 2

np.array_equal(np.array(loop_result), numpy_result)

True

✅ **Your Turn**: Time `sum(range(10_000))` against `np.sum(np.arange(10_000))`. Then time `np.sum(arr)` where `arr = np.arange(10_000)` was created **beforehand**. How much of NumPy's time was spent just creating the array? In a comment, explain which comparison is fair for which question.

In [41]:
%timeit sum(range(10_000))
%timeit np.sum(np.arange(10_000))

arr = np.arange(10_000)
%timeit np.sum(arr)

113 μs ± 1.63 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
6.23 μs ± 62.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
3.24 μs ± 107 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


1 vs 2 both start from nothing, so they are the most fair comparison

## 6. Python Loops vs. NumPy
This is the payoff. NumPy (notebook 07) does the same calculation **in bulk**:
- **Python loop:** Python handles one value at a time, checking its type and looking up `**` on every pass.
- **NumPy (vectorized):** Python makes **one** call, and NumPy runs the whole loop in fast, compiled code.

### Squaring

In [42]:
numbers_list = list(range(1_000_000))
numbers_array = np.arange(1_000_000)

def square_with_loop(values):
    result = []
    for value in values:
        result.append(value ** 2)
    return result

def square_with_numpy(values):
    return values ** 2

t_loop = %timeit -o square_with_loop(numbers_list)
t_numpy = %timeit -o square_with_numpy(numbers_array)
print(f'NumPy is about {t_loop.average / t_numpy.average:.0f}x faster')

59.4 ms ± 672 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.52 ms ± 33.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
NumPy is about 39x faster


### Filtering
A **boolean mask** (`values[values >= 700_000]`, from notebook 07) replaces a loop with an `if` inside:

In [43]:
def passing_loop(values):
    result = []
    for score in values:
        if score >= 700_000:
            result.append(score)
    return result

def passing_numpy(values):
    return values[values >= 700_000]

%timeit passing_loop(numbers_list)
%timeit passing_numpy(numbers_array)

17.7 ms ± 415 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
679 μs ± 11.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Summary Statistics

In [44]:
def mean_with_loop(values):
    total = 0
    for value in values:
        total += value
    return total / len(values)

%timeit mean_with_loop(numbers_list)
%timeit np.mean(numbers_array)

22.3 ms ± 1.17 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
567 μs ± 19.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Speed isn't the only win. `values[values >= 700_000]` is also **shorter and clearer** than six lines of loop. It says *what* you want rather than *how* to do it.

⚠️ Looping **over a NumPy array** with a plain `for` loop is even slower than looping over a list, because each element is converted to a Python object one at a time. Arrays are fast only when NumPy does the looping.

✅ **Your Turn**:
1. Write a loop that adds 10 to every value in `numbers_list`, and a NumPy version for `numbers_array`. Check that the results match, then time both.
2. Do the same for selecting the values **greater than 500,000**.

In [48]:
def add10_loop(values):
    result = []
    for v in values:
        result.append(v + 10)
    return result

def add10_numpy(values):
    return values + 10

print('Match:', np.array_equal(np.array(add10_loop(numbers_list)), add10_numpy(numbers_array)))
%timeit add10_loop(numbers_list)
%timeit add10_numpy(numbers_array)

def above_loop(values):
    result = []
    for v in values:
        if v > 500_000:
            result.append(v)
    return result

def above_numpy(values):
    return values[values > 500_000]

print('Match:', np.array_equal(np.array(above_loop(numbers_list)), above_numpy(numbers_array)))
%timeit above_loop(numbers_list)
%timeit above_numpy(numbers_array)

Match: True
46.4 ms ± 1.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.47 ms ± 34.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Match: True
21.8 ms ± 594 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
908 μs ± 17.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 7. Repeated Work and How Runtime Grows
A very common slowdown is recalculating the **same thing inside a loop**. Here, every score is divided by the top score, but `max(scores)` is recomputed on **every** pass:

In [46]:
def normalize_slow(scores):
    return [s / max(scores) for s in scores]   # max() scans the whole list, every time

def normalize_fast(scores):
    top = max(scores)                          # compute it once
    return [s / top for s in scores]

scores = list(np.random.randint(0, 100, size=2_000))
print(normalize_slow(scores) == normalize_fast(scores))   # same result?

%timeit normalize_slow(scores)
%timeit normalize_fast(scores)

True
45.9 ms ± 2.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
126 μs ± 2.31 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Watch How It Scales
Now **double** the data a few times. `normalize_fast` takes about **2x** as long each time (it's *linear*). `normalize_slow` takes about **4x** as long, because it does *n* passes of *n* steps each (it's *quadratic*). On large data, that difference is everything:

In [47]:
for n in [1_000, 2_000, 4_000]:
    data = list(range(1, n + 1))
    t_slow = %timeit -o -q -n 3 -r 3 normalize_slow(data)
    t_fast = %timeit -o -q -n 3 -r 3 normalize_fast(data)
    print(f'n={n:>5}:  slow {t_slow.average * 1000:8.2f} ms   fast {t_fast.average * 1000:6.3f} ms')

n= 1000:  slow     7.55 ms   fast  0.036 ms
n= 2000:  slow    27.98 ms   fast  0.071 ms
n= 4000:  slow   112.72 ms   fast  0.144 ms


(`-q` keeps `%timeit` quiet, so only the `print` shows.)

Ask yourself: *Am I recalculating the same max, filter, conversion, or **file read** inside a loop?* If so, do it once, before the loop.

✅ **Your Turn**: Write a loop that recomputes something it doesn't need to, such as `len(my_list)` or `sum(my_list)` inside every pass. Move it outside the loop, and use `%timeit` to compare the two versions.

In [49]:
def center_slow(values):
    return [v - sum(values) / len(values) for v in values]   # mean recomputed every pass

def center_fast(values):
    mean = sum(values) / len(values)                          # computed once
    return [v - mean for v in values]

my_list = list(range(2_000))
print('Same result:', center_slow(my_list) == center_fast(my_list))

%timeit center_slow(my_list)
%timeit center_fast(my_list)

Same result: True
21.3 ms ± 1.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
90.6 μs ± 12.1 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


✅ **Your Turn**: Try squaring numbers with a Python loop, a list comprehension, and NumPy at several sizes (say 1,000, 100,000, and 1,000,000). How does the **gap** between them change as the size grows?

In [50]:
for size in [1_000, 100_000, 1_000_000]:
    as_list = list(range(size))
    as_array = np.arange(size)

    t_loop = %timeit -o -q -n 3 -r 3 square_with_loop(as_list)
    t_comp = %timeit -o -q -n 3 -r 3 [v ** 2 for v in as_list]
    t_np = %timeit -o -q -n 3 -r 3 as_array ** 2

    print(f'n={size:>9,}: loop {t_loop.average*1000:9.3f} ms | comp {t_comp.average*1000:9.3f} ms | '
          f'numpy {t_np.average*1000:7.3f} ms | numpy is {t_loop.average / t_np.average:5.0f}x faster than the loop')

n=    1,000: loop     0.045 ms | comp     0.056 ms | numpy   0.017 ms | numpy is     3x faster than the loop
n=  100,000: loop     4.714 ms | comp     3.972 ms | numpy   0.046 ms | numpy is   101x faster than the loop
n=1,000,000: loop    57.524 ms | comp    55.832 ms | numpy   1.437 ms | numpy is    40x faster than the loop


All of the values increase slightly more than the ratio of n

## 8. Finding the Slow Part: Profiling
`%time` and `%timeit` tell you how long a whole block takes. A **profiler** tells you **where** inside it the time goes. `%prun` runs a statement and reports the time spent in each function it called.

The pipeline below calls two helper functions for every value. Which one is the bottleneck?

In [51]:
def double(value):
    return value * 2

def is_banned(value, banned):
    return value in banned          # searching a list checks items one by one

def pipeline(values, banned):
    kept = []
    for v in values:
        if not is_banned(v, banned):
            kept.append(double(v))
    return kept

banned_list = list(range(0, 20_000, 7))
%prun -s cumulative -l 6 pipeline(range(20_000), banned_list)

         54347 function calls in 0.276 seconds

   Ordered by: cumulative time
   List reduced from 40 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    20000    0.264    0.000    0.264    0.000 3096098862.py:4(is_banned)
        1    0.000    0.000    0.258    0.258 {built-in method builtins.exec}
        1    0.000    0.000    0.257    0.257 <string>:1(<module>)
        1    0.008    0.008    0.257    0.257 3096098862.py:7(pipeline)
    17142    0.002    0.000    0.002    0.000 {method 'append' of 'list' objects}
    17142    0.001    0.000    0.001    0.000 3096098862.py:1(double)

How to read it:
- **ncalls**: how many times the function was called
- **tottime**: time spent **inside** that function itself
- **cumtime**: time inside it **plus** everything it called

`is_banned` dominates. Searching a **list** with `in` checks items one at a time, but a **set** finds an item almost instantly. That's a one-word fix:

In [52]:
banned_set = set(banned_list)

print(pipeline(range(20_000), banned_list) == pipeline(range(20_000), banned_set))
%timeit pipeline(range(20_000), banned_list)
%timeit pipeline(range(20_000), banned_set)

True
274 ms ± 6.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
1.79 ms ± 22.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


(`-s cumulative` sorts by cumulative time and `-l 6` shows only the top 6 rows. In VS Code, the report appears in the cell's output.)

Profile only code you've **already measured** to be slow. Profiling every two-line example is like using a metal detector to look for a paperclip.

### Line by Line (Optional)
When `%prun` points at one function but not at the slow **line**, the `line_profiler` package can time each line separately. It isn't part of the course requirements, so install it first with `%pip` (notebook 01):

In [53]:
%pip install line_profiler

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
%load_ext line_profiler
%lprun -f pipeline pipeline(range(20_000), banned_list)

Timer unit: 1e-07 s

Total time: 0.325817 s
File: C:\Users\bpoec\AppData\Local\Temp\ipykernel_79732\3096098862.py
Function: pipeline at line 7

Line #      Hits         Time  Per Hit   % Time  Line Contents
     7                                           def pipeline(values, banned):
     8         1         19.0     19.0      0.0      kept = []
     9     20001      51173.0      2.6      1.6      for v in values:
    10     20000    3078264.0    153.9     94.5          if not is_banned(v, banned):
    11     17142     128711.0      7.5      4.0              kept.append(double(v))
    12         1          2.0      2.0      0.0      return kept

The `% Time` column shows which line is the bottleneck. If you can't install `line_profiler`, split the function into pieces and `%timeit` each one.

✅ **Your Turn**: Write a function that runs **two** loops, one cheap (for example, summing a list) and one expensive (for example, checking whether each of 5,000 numbers is `in` a 5,000-item list). Use `%prun` to show which part takes the most time.

In [55]:
def two_loops(values, lookup):
    total = 0
    for v in values:        
        total += v

    found = 0
    for v in values:          
        if v in lookup:
            found += 1
    return total, found

nums = list(range(5_000))
lookup_list = list(range(2_500, 7_500))

%prun -s cumulative -l 6 two_loops(nums, lookup_list)

         64 function calls in 0.070 seconds

   Ordered by: cumulative time
   List reduced from 37 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.054    0.054 {built-in method builtins.exec}
        1    0.000    0.000    0.054    0.054 <string>:1(<module>)
        1    0.053    0.053    0.053    0.053 3835757037.py:1(two_loops)
        2    0.016    0.008    0.016    0.008 {method '__exit__' of 'sqlite3.Connection' objects}
        2    0.000    0.000    0.000    0.000 traitlets.py:726(__set__)
        2    0.000    0.000    0.000    0.000 traitlets.py:3627(set)

## 9. Measuring Memory
Fast code can still be a problem if it uses too much memory. NumPy arrays tell you exactly how much space their data takes:

In [56]:
values = np.arange(1_000_000)

print('shape:', values.shape)
print('dtype:', values.dtype)
print('bytes:', f'{values.nbytes:,}')    # 1,000,000 values x 8 bytes each

shape: (1000000,)
dtype: int64
bytes: 8,000,000


The **dtype** decides the size of each value. If your numbers fit in a smaller type, converting with `.astype()` cuts memory, but check the range first! `int8` can only hold -128 to 127.

In [57]:
for dtype in ['int64', 'int32', 'int16', 'float32']:
    print(f'{dtype:8} {values.astype(dtype).nbytes:>10,} bytes')

int64     8,000,000 bytes
int32     4,000,000 bytes
int16     2,000,000 bytes
float32   4,000,000 bytes


### Lists Are Bigger Than They Look
`sys.getsizeof()` reports an object's size, but for a list it counts only the list's internal "pointers", **not** the numbers themselves. Treat it as a starting point, not a full memory audit:

In [58]:
as_list = list(range(1_000_000))

print(f'list (pointers only):    {sys.getsizeof(as_list):>12,} bytes')
print(f'list (with the ints):    {sys.getsizeof(as_list) + sum(sys.getsizeof(n) for n in as_list):>12,} bytes')
print(f'NumPy array (int64):     {values.nbytes:>12,} bytes')

list (pointers only):       8,000,056 bytes
list (with the ints):      36,000,056 bytes
NumPy array (int64):        8,000,000 bytes


✅ **Your Turn**: Create a NumPy array of one million random **floats** with `np.random.random(1_000_000)`. Check its `shape`, `dtype`, and `nbytes`. Then convert it to `float32` and compare the memory. Print the first 3 values of each. What did you give up for the smaller size?

In [59]:
floats = np.random.random(1_000_000)
print('shape:', floats.shape)
print('dtype:', floats.dtype)
print('bytes:', f'{floats.nbytes:,}')

floats32 = floats.astype('float32')
print('float32 bytes:', f'{floats32.nbytes:,}')

print(floats[:3])
print(floats32[:3])

shape: (1000000,)
dtype: float64
bytes: 8,000,000
float32 bytes: 4,000,000
[0.83657845 0.81184304 0.62708324]
[0.8365784  0.81184304 0.62708324]


## 10. Working with Data Efficiently
The same ideas apply to Pandas (notebook 06). First, let's make a realistic-sized CSV file of 100,000 sales records:

In [60]:
np.random.seed(3603)
n = 100_000
sales = pd.DataFrame({
    'date': pd.date_range('2026-01-01', periods=n, freq='min').strftime('%Y-%m-%d'),
    'department': np.random.choice(['Books', 'Electronics', 'Home', 'Sports'], size=n),
    'store_id': np.random.randint(1, 50, size=n),
    'units': np.random.randint(1, 10, size=n),
    'amount': np.random.uniform(5, 500, size=n).round(2),
    'notes': np.random.choice(['', 'gift', 'returned', 'online order'], size=n),
})
sales.to_csv(OUT / 'sales.csv', index=False)
sales.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   date        100000 non-null  str    
 1   department  100000 non-null  str    
 2   store_id    100000 non-null  int32  
 3   units       100000 non-null  int32  
 4   amount      100000 non-null  float64
 5   notes       100000 non-null  str    
dtypes: float64(1), int32(2), str(3)
memory usage: 17.7 MB


💡 `memory_usage='deep'` counts the real size of text columns. Without it, Pandas only estimates them.

### Read Only the Columns You Need
`usecols` (notebook 06) skips the other columns entirely, which saves both time and memory:

In [61]:
%time full = pd.read_csv(OUT / 'sales.csv')
%time slim = pd.read_csv(OUT / 'sales.csv', usecols=['department', 'amount'])

print(f'full: {full.memory_usage(deep=True).sum():>12,} bytes')
print(f'slim: {slim.memory_usage(deep=True).sum():>12,} bytes')

CPU times: total: 46.9 ms
Wall time: 57.6 ms
CPU times: total: 46.9 ms
Wall time: 35 ms
full:   18,925,944 bytes
slim:    6,349,762 bytes


### Use `category` for Repeated Text
A text column with only a few distinct values, like `department`, can be stored as a **category**. Each distinct value is stored once, and every row just holds a small code:

In [62]:
as_text = pd.read_csv(OUT / 'sales.csv', usecols=['department'])
as_category = pd.read_csv(OUT / 'sales.csv', usecols=['department'], dtype={'department': 'category'})

print(f'text:     {as_text.memory_usage(deep=True).sum():>10,} bytes')
print(f'category: {as_category.memory_usage(deep=True).sum():>10,} bytes')

text:      5,549,762 bytes
category:    100,526 bytes


### Process Huge Files in Chunks
If a file is too big to fit in memory, `chunksize` reads it a piece at a time, so you can keep a running total:

In [63]:
total_revenue = 0
for chunk in pd.read_csv(OUT / 'sales.csv', chunksize=25_000):
    total_revenue += chunk['amount'].sum()

print(f'Total revenue: ${total_revenue:,.2f}')
print('Check:         $' + f"{full['amount'].sum():,.2f}")

Total revenue: $25,281,028.41
Check:         $25,281,028.41


### Avoid Row-by-Row Loops on DataFrames
`.iterrows()` loops over a DataFrame one row at a time, which is the Pandas version of the slow Python loop. Whole-column operations are vectorized, just like NumPy:

In [64]:
%%time
discounted = []
for _, row in full.iterrows():
    discounted.append(row['amount'] * 0.90)

CPU times: total: 1.48 s
Wall time: 1.48 s


In [65]:
%%time
full['discounted'] = full['amount'] * 0.90

CPU times: total: 0 ns
Wall time: 956 μs


✅ **Your Turn**:
1. Read `sales.csv` with `usecols` for just `store_id` and `units`. Compare its time and memory to `full`.
2. Use `.iterrows()` to compute `units * amount` for every row (on `full.head(10_000)`, to keep it quick), then do the same thing with one column operation. Time both and check that the results match.

In [67]:
%time slim2 = pd.read_csv(OUT / 'sales.csv', usecols=['store_id', 'units'])
print(f'full:  {full.memory_usage(deep=True).sum():>12,} bytes')


CPU times: total: 46.9 ms
Wall time: 43.6 ms
full:    19,725,944 bytes


In [70]:
%%time
sample = full.head(10_000)
row_totals = []
for _, row in sample.iterrows():
    row_totals.append(row['units'] * row['amount'])

    

CPU times: total: 156 ms
Wall time: 156 ms


In [71]:
%%time
col_totals = sample['units'] * sample['amount']

CPU times: total: 0 ns
Wall time: 404 μs


## 11. Trustworthy Timings
Notebook timings can mislead when cells depend on leftover variables, or when something was **cached** by an earlier run. For results you'll report:
1. **Restart** the kernel (notebook 02).
2. Run the **setup** cells.
3. Run the **timing** cell, then repeat the measurement.
4. Record the **typical** result, not the luckiest one.

Also, keep setup **out** of the timed line. `%timeit np.sqrt(values)` measures the square root, not the creation of `values`.

✅ **Your Turn**: Restart the kernel, re-run the Setup cell and the squaring comparison in section 6, and compare the new speedup to your first run. In the cell below, explain why restarting makes results more trustworthy, and why the two numbers aren't identical.

Restarting clears leftover variables so every measurement starts from the same conditions a grader would have. The two speeds aren't identical because background programs, CPU clock speed changes, and memory caching vary from run to run.

---
### Summary
- **Measure, don't guess.** Find the bottleneck, improve it, then measure again.
- `%time`/`%%time` give a quick single measurement (wall time is what you waited). `%timeit`/`%%timeit` run many times for reliable comparisons. Use `-o` to save the result and compute a speedup.
- Variables created inside `%%timeit` don't survive. `%%time` keeps them.
- Compare fairly: same input, same work, same result, setup excluded.
- NumPy and Pandas **vectorized** operations beat Python loops on large data, and are often clearer too. Never loop over an array or use `.iterrows()` when a whole-column operation exists.
- Move repeated work out of loops. Watch for runtime that grows faster than the data.
- `%prun` finds the slow function, and `%lprun` finds the slow line. Lists search slowly and sets search fast.
- Check memory with `.nbytes`, `.astype()`, and `memory_usage(deep=True)`. Load less data with `usecols`, `category`, and `chunksize`.

## 🏋️ Practice

### Practice 1 — Time a Sort (easy)
Create a list of 100,000 random numbers (hint: `random.sample(range(1_000_000), 100_000)` after `import random`) and use `%timeit` to measure how long `sorted(...)` takes on it.

In [74]:
import random
random_numbers = random.sample(range(1_000_000), 100_000)
%timeit sorted(random_numbers)

19.3 ms ± 1.08 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Practice 2 — Loop vs. Comprehension (medium)
Using the `numbers` list below, compare `%timeit` for a `for` loop that builds a squared list with `.append()` vs. a list comprehension that does the same thing. Check that both give the same list, and use `-o` to print how many times faster the winner is.

In [75]:
numbers = list(range(1_000))
def squares_loop(values):
    result = []
    for n in values:
        result.append(n ** 2)
    return result

print('Same list:', squares_loop(numbers) == [n ** 2 for n in numbers])

t_loop = %timeit -o squares_loop(numbers)
t_comp = %timeit -o [n ** 2 for n in numbers]
print(f'The comprehension is {t_loop.average / t_comp.average:.2f}x faster')


Same list: True
40.4 μs ± 1.99 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
33.2 μs ± 807 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
The comprehension is 1.22x faster


### Practice 3 — Three-Way Comparison (harder)
Compare `%timeit` for squaring 100,000 numbers three ways: a `for` loop over `list(range(100_000))`, a list comprehension over the same list, and NumPy on `np.arange(100_000)`. Which is fastest? Which is slowest? Then repeat the loop version over the **NumPy array** instead of the list, and explain the result (see section 6).

In [76]:
big_list = list(range(100_000))
big_array = np.arange(100_000)

t_loop = %timeit -o square_with_loop(big_list)
t_comp = %timeit -o [v ** 2 for v in big_list]
t_np = %timeit -o big_array ** 2
t_loop_arr = %timeit -o square_with_loop(big_array)

print(f'loop over list:  {t_loop.average*1000:8.3f} ms')
print(f'comprehension:   {t_comp.average*1000:8.3f} ms')
print(f'NumPy:           {t_np.average*1000:8.3f} ms')
print(f'loop over array: {t_loop_arr.average*1000:8.3f} ms')
# Fastest: NumPy. Slowest: the for loop over the NumPy array.

4.96 ms ± 269 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.24 ms ± 171 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
42.8 μs ± 1.16 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
6.37 ms ± 116 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
loop over list:     4.965 ms
comprehension:      4.235 ms
NumPy:              0.043 ms
loop over array:    6.373 ms


## 🔥 Challenge (Optional) — Where's the Line?
For very small inputs, a plain Python loop can actually be just as fast as (or faster than) NumPy, because NumPy has a small fixed overhead per call. Find roughly how small an array has to be before the loop wins, by timing both approaches at a few different sizes (e.g. 10, 100, 1,000, 100,000).

In [ ]:
# Your investigation here


## 📦 Check Your Output Files
This notebook's only output file is `sales.csv`, the generated practice data. Upload `08-OutputFiles` along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path, f'({path.stat().st_size:,} bytes)')